<a href="https://colab.research.google.com/github/StevenPeutz/Disinformation-NLP/blob/master/MAIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About this Notebook
This is the full notebook <sup>1</sup> that provides all embeddings and models compaired throughout this project. This means a 6 x 7 x 4 design in terms of embeddings, models and testsets resulting in 42 NLP architectures, each tested on 4 noise<sup>2</sup> levels.   
<br>

**Embeddings:**
- CountVectorizer (BoW)
- HashingVector (HV)
- TF-IDF
- GloVe<sup>3</sup>
- Word2Vec<sup>4</sup> (w2v)
- FastText<sup>5</sup> (ft)
<br><br>

**Classification Models**
- Logistic Regression (LR)
- Naive Bayes (NB)
- Random Forest (RF)
- Support Vector Machine (SVM)
- K-Nearest Neighbour (KNN)
- GradientBoosting (GB)
- Extreme Gradient Boosting (XGB)   
  
<br>


---


<br>
  
<sup>1  </sup>*In order for all this to fit within the 35GB RAM bounds of the google colab environment, the pretrained embeddings (GloVe, w2v and fasttext) used in this notebook have undergone three restrictions (1.reduced to 50 dimensions per word (for w2v and fasttext this was done through CPA), 2. the 'maxlength' of each doc has been reduced to 80 tokens, and 3. the train set used for training takes a sample of 20k docs instead of the full 51k).  Full embeddings, models and their results can be found in the separate notebooks.* *italicised text*    
<br>
  

<sup>2  </sup>*Noise levels as introduced by machine backtranslations. 'N0' being the original testset (the version similar to training), 'N1' (1 level of backtranslation (EN -> RU -> EN), continuing to 'N3'.
For rough assessment of noise in a purely lexical sense, the Jaro-Winkler Distances (normalized) have been calculated and imported before and are imported in this notebook*. 
<br>
<br>


<sup>3,4,5  </sup>*The reduced embeddings (FastText, Word2Vec and GloVe) created in the 'DimReduction Notebook' can be downloaded as gzipped txt files from the following public kaggle destination:*  <br>
<br>

|                Embedding               | Dims | Size (gzipped) |  Storage   |                                                 Download link                                                 |
| :------------------------------------|:----:|:--------------:|:----------:|:---------------------------------------------------------------------------------------------------------: |
|           GloVe  (50dim version)          | 50   |   66 MB        |   Kaggle   |         https://www.kaggle.com/datasets/stevenpeutz/tinypretrainedembeddings         |
|      FastText - Reduced (PCA)     | 50   |   240 MB       |   Kaggle   |         https://www.kaggle.com/datasets/stevenpeutz/tinypretrainedembeddings         |
|     Word2Vec - Reduced (PCA)       | 50   |   958 MB       |   Kaggle   |         https://www.kaggle.com/datasets/stevenpeutz/tinypretrainedembeddings         |


<br>


# Import + mounting drive

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse.csc import csc_matrix
import matplotlib.pyplot as plt

#preprocessing:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.preprocessing import MaxAbsScaler
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences #or: from tensorflow.keras.preprocessing.sequence import pad_sequences
#import nltk #not needed as now I am importing the stemmed train and test sets directly.


#pickle:
from pickle import dump, load
picklepath = '/content/drive/MyDrive/MYDATA/PickledModels/MAIN/'

#classifiers:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

#evaluation
from sklearn.metrics import roc_auc_score


#system:
import gc
import sys
from google.colab import drive
import gzip

In [ ]:
#from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#no need to run this everytime
'''
!pip install pipreqs
!pipreqs path_here
'''
#pipreqs is great but doesnt seem to work for ipynb..
#Solution: convert notebook to .py file (copy) can be done using builtin google colab functionality or with the nbconvert packlage 
#and then running pipreqs on that. This was done using the terminal.

In [ ]:
# this colab builds on the FULL_CREATION_DataSet.ipynb colab
# link: https://colab.research.google.com/drive/1lFMu0WaWkpDCZpPCH3eC3LJWlYNlbg06?usp=sharing
# this previous colab exports the seperate train and test sets..

# Reading in test & train sets (csv method)




In [ ]:
pathTESTcsv = '/content/drive/MyDrive/MYDATA/DATA/full_TEST_Nx_stemmed.csv'
pathTRAINcsv = '/content/drive/MyDrive/MYDATA/DATA/full_TRAIN_stemmed.csv'

df_test = pd.read_csv(pathTESTcsv)
df_train = pd.read_csv(pathTRAINcsv)

In [ ]:
df_test.head()

In [ ]:
#BE AWARE:
df_train = df_train.sample(30000, random_state=42) #smaller for faster testing..

In [ ]:
len(df_test[df_test['text_N1'].str.contains('?|?|?')]) #how many rows are not translated back to english correctly? # ?30%..
# redo preceeding colab, but with more rigorous cleaning (en see if thats helps), if not, redo with other languaage than russian.

# Embedding 0: Assigning the series needed for embedding


*   X_train
*   y_train
*   X_testNx (for N0 to N3)
*   y_test (same for all N's)

In [ ]:
X_train = df_train['text']
y_train = df_train['label']
X_testN0 = df_test['text']
X_testN1 = df_test['text_N1']
X_testN2 = df_test['text_N2']
X_testN3 = df_test['text_N3']
y_testALL = df_test['label']

In [ ]:
#X_train[:10]
X_train.shape

# Embedding 1: BoW (Countvectorizer)

In [ ]:
#from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(min_df=4, max_df=0.8)
vectorizer.fit(X_train)

In [ ]:
X_train_BoW = vectorizer.transform(X_train)
# ---
X_testN0_BoW  = vectorizer.transform(X_testN0)
X_testN1_BoW  = vectorizer.transform(X_testN1)
X_testN2_BoW  = vectorizer.transform(X_testN2)
X_testN3_BoW  = vectorizer.transform(X_testN3)

In [ ]:
#X_train.shape # <- 67625 by 1
#X_train_BoW.shape # <- 67625 by 60326
#type(X_testN0_BoW) # <- sparse matrix
#type(X_train) #<- series

In [ ]:
#n_features = len(X_train)
#print(n_features)

# Embedding 2: HashingVectorizer

In [ ]:
#from sklearn.feature_extraction.text import HashingVectorizer
hashingvectorizer = HashingVectorizer(n_features=60326)
hashingvectorizer.fit(X_train)

In [ ]:
X_train_HV = hashingvectorizer.transform(X_train)
# ---
X_testN0_HV  = hashingvectorizer.transform(X_testN0)
X_testN1_HV  = hashingvectorizer.transform(X_testN1)
X_testN2_HV  = hashingvectorizer.transform(X_testN2)
X_testN3_HV  = hashingvectorizer.transform(X_testN3)

In [ ]:
#THIS BLOCK IS ONLY FOR THE NAIVE BAYES CLASSIFIER

#scaling
#minmax would be the go to for sparse data, however our sparse data contains negative values -> MaxAbsScaler

#from sklearn.preprocessing import MaxAbsScaler
MaxAbs_scaler = MaxAbsScaler()

# transform data
# (check whether indeed best to fit scaler only on trainset)
MaxAbs_scaler.fit(X_train_HV)
X_train_HV_scaled = MaxAbs_scaler.transform(X_train_HV)
X_testN0_HV_scaled = MaxAbs_scaler.transform(X_testN0_HV)
X_testN1_HV_scaled = MaxAbs_scaler.transform(X_testN1_HV)
X_testN2_HV_scaled = MaxAbs_scaler.transform(X_testN2_HV)
X_testN3_HV_scaled = MaxAbs_scaler.transform(X_testN3_HV)

#shift +1 for all non-zero values to be usable for Naive Bayes classifier
#from scipy.sparse.csc import csc_matrix
X_train_HV_scaled.data += 1
X_testN0_HV_scaled.data += 1
X_testN1_HV_scaled.data += 1
X_testN2_HV_scaled.data += 1
X_testN3_HV_scaled.data += 1

# Embedding 3: TF-IDF
- Builds on output of BoW countvectorizer (embedding 1)

In [ ]:
#from sklearn.feature_extraction.text import TfidfTransformer
tfidf_transformer = TfidfTransformer()
tfidf_transformer.fit(X_train_BoW)

In [ ]:
X_train_TFIDF = tfidf_transformer.transform(X_train_BoW)
# ---
X_testN0_TFIDF  = tfidf_transformer.transform(X_testN0_BoW)
X_testN1_TFIDF  = tfidf_transformer.transform(X_testN1_BoW)
X_testN2_TFIDF  = tfidf_transformer.transform(X_testN2_BoW)
X_testN3_TFIDF  = tfidf_transformer.transform(X_testN3_BoW)

In [ ]:
#X_testN0_TFIDF.shape # <- 684 by 60326
#type(X_testN0_TFIDF) # <- sparse matrix
X_testN0_TFIDF.shape

# Prepping for pretraind embeddings
- GloVe
- w2v
- fasttext

In [ ]:
#import nltk
#from keras.preprocessing.text import Tokenizer
#from keras_preprocessing.sequence import pad_sequences

Separate collab using a small test set to try to work out this problem: https://colab.research.google.com/drive/1Pi55fZS9rlYyouN1bzi47lNkiSB2PzBI?usp=sharing 

//at this point 2.2GB should be used.


For other pretrained embeddings (also 50dim ones):
https://developer.syn.co.in/tutorial/bot/oscova/pretrained-vectors.html 

In [ ]:
#3.6GB prior to this block
word_tokenizer = Tokenizer(num_words = 1000)
word_tokenizer.fit_on_texts(X_train)

word_index = word_tokenizer.word_index
vocab_length = len(word_index) + 1

In [ ]:
#list(word_index)[-300:]

#as I thought, the vocab can be made lot shorter by:
# - cleaning data (removing " before tokenizing)
# - removing @mentions or somehting (perhaps just remove everything immediately following @?)

In [ ]:
print(word_index.get('trump'))
# so this dictionary (word_index) has the vocab (267981 words..) stored sort of in order of usage ('the' is 1, 'a' is 4)

In [ ]:
#convert series to list (prior to this 2.7GB used)
X_train_sequences = word_tokenizer.texts_to_sequences(X_train)
# ---
X_testN0_sequences = word_tokenizer.texts_to_sequences(X_testN0)
X_testN1_sequences = word_tokenizer.texts_to_sequences(X_testN1)
X_testN2_sequences = word_tokenizer.texts_to_sequences(X_testN2)
X_testN3_sequences = word_tokenizer.texts_to_sequences(X_testN3)

In [ ]:
# Output the results to inspect what is happening
print("Word index for example word 'home' :\n", word_index.get('home')) #chosen as it is the 1st word in X_train 1st doc
print("\nTraining sequence of first doc:\n", X_train_sequences[0]) #first doc, starting with word 'home'
print("\nX_train_sequences data type:", type(X_train_sequences))

In [ ]:
# Get max training sequence length
maxlenCalc = max([len(x) for x in X_train_sequences])
print(maxlenCalc)

In [ ]:
#set some restrictive limits to make it fast for now
#max_features = 500
max_len = 80 #replace with: maxlenCalc = max([len(x) for x in X_train_sequences])
              #or replace with desired length of dim divided by 50 (length of vectors per word)
              #For now limited to a mere 80 to save RAM in order to fit all in a single 35GB RAM environment
padding_type='post'
truncation_type='post'

In [ ]:
#padding or truncating the lists (sequences) to make equal length. Now Numpy Arrays
# replace with maxlenCalc (887), but keep at 300 for now for speed..
X_train_SeqPad = pad_sequences(X_train_sequences, maxlen=max_len, padding=padding_type, truncating=truncation_type)
# ---
X_testN0_SeqPad = pad_sequences(X_testN0_sequences, maxlen=max_len, padding=padding_type, truncating=truncation_type)
X_testN1_SeqPad = pad_sequences(X_testN1_sequences, maxlen=max_len, padding=padding_type, truncating=truncation_type)
X_testN2_SeqPad = pad_sequences(X_testN2_sequences, maxlen=max_len, padding=padding_type, truncating=truncation_type)
X_testN3_SeqPad = pad_sequences(X_testN3_sequences, maxlen=max_len, padding=padding_type, truncating=truncation_type)

#after this block: 3.0gb

In [ ]:
# Output the results to inspect what is happening
print("\nPadded training sequences of first doc:\n", X_train_SeqPad[0])
print("\nX_train_SeqPad data type:", type(X_train_SeqPad))
print("\nX_train_SeqPad shape:", X_train_SeqPad.shape) #rows (67625) by maxlen (110)

# Embedding 4: GloVe

In [ ]:
#EMBEDDING_INDEX

# 4GB RAM used prior to this point
#Creating the GloVe Embedding Index (aka embedding dictionary)
#This does not use anything else than the GloVe data (words and their vectors)

embedding_dim = 50 #based on glovefile..  ('...50d.txt')
embeddings_index = {} # == 'embeddings_dictionary' in https://www.kaggle.com/code/andreshg/nlp-glove-bert-tf-idf-lstm-explained#5.-Vectorization

with gzip.open('/content/drive/MyDrive/MYDATA/Embeddings_PreTrained/glove.6B/glove.6B.50d.txt.gz', 'r') as f:
    for line in f:
        values = line.split()
        word = values[0].decode('utf-8')
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f'Found {len(embeddings_index)} word vectors.')
#note to self: the dim number (50) does not influence the total word vector count (i.e. 400k for either 50 or 100 dims..)

In [ ]:
#type(embeddings_index) # <- dict
print(embeddings_index["trump"])
#len(embeddings_index["trump"]) # =50

In [ ]:
#EMBEDDING_MATRIX
#this is where we need to grab each word from the padded sequences and match them with the embeddings index to create the embedding_matrix
embedding_matrix = np.zeros((vocab_length, embedding_dim))

for word, embedding in word_tokenizer.word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[embedding] = embedding_vector
        
#embedding_matrix #numpy array

In [ ]:
embedding_matrix[1]

In [ ]:
# replacing the above 5 blocks:
# this block jumps from 4GB to 15.5GB..

def embedding_generator(X, embedding_matrix):
    for x in X:
        yield embedding_matrix[x]
#(the generator function will be reused for wrod2vec and fasttext)

# training set
nparraylist_train = np.array(list(embedding_generator(X_train_SeqPad, embedding_matrix)))

# test sets
test_sets = [X_testN0_SeqPad, X_testN1_SeqPad, X_testN2_SeqPad, X_testN3_SeqPad]
nparraylist_tests = []
for X_test in test_sets:
    nparraylist_tests.append(np.array(list(embedding_generator(X_test, embedding_matrix))))

# reshape arrays
dims = (len(nparraylist_train), len(nparraylist_train[0]) * len(nparraylist_train[0][0]))
X_train_GloVe = np.reshape(nparraylist_train, dims)

X_tests_GloVe = []
for nparraylist_test in nparraylist_tests:
    dims = (len(nparraylist_test), len(nparraylist_test[0]) * len(nparraylist_test[0][0]))
    X_tests_GloVe.append(np.reshape(nparraylist_test, dims))

X_testN0_GloVe, X_testN1_GloVe, X_testN2_GloVe, X_testN3_GloVe = X_tests_GloVe


In [ ]:
#type(nparraylist_train) #numpy.ndarray
#nparraylist_train.shape # (51331, 180, 50)
np.array_equal(X_testN0_GloVe, X_testN1_GloVe, equal_nan=False) #False, so that seems correct

In [ ]:
#14.8RAM
#viewing if indeed corret:

#len(nparraylist_tests[3]) #21999. (0 (N0), 1 (N1), 2 (N2),  and 3 (N3))
#nparraylist_tests[3].shape

In [ ]:
"""
nparraylist_train, nparraylist_testN0, nparraylist_testN1, nparraylist_testN2, nparraylist_testN3 = [], [], [], [], []
del nparraylist_train, nparraylist_testN0, nparraylist_testN1, nparraylist_testN2, nparraylist_testN3
"""

In [ ]:
#df_train, embedding_matrix, X_train_SeqPad, X_testN0_SeqPad, X_testN1_SeqPad, X_testN2_SeqPad, X_testN3_SeqPad, embeddings_index, df_test, word_index, X_train_sequences, X_testN0_sequences, X_testN1_sequences, X_testN2_sequences, X_testN3_sequences = [], [], [], [], [], [], [], [], [], [], [], [], [], [], []
#del df_train, X_train, embedding_matrix, X_train_SeqPad, X_testN0_SeqPad, X_testN1_SeqPad, X_testN2_SeqPad, X_testN3_SeqPad, embeddings_index, df_test, word_index, X_train_sequences, X_testN0_sequences, X_testN1_sequences, X_testN2_sequences, X_testN3_sequences

In [ ]:
#gc.collect()

In [ ]:
#THIS BLOCK IS ONLY FOR THE NAIVE BAYES CLASSIFIER

#scaling
#minmax would be the go to for sparse data, however our sparse data contains negative values -> MaxAbsScaler
#from sklearn.preprocessing import MaxAbsScaler
#THis takes another 10GB...

MaxAbs_scaler = MaxAbsScaler()

# transform data
# (check whether indeed best to fit scaler only on trainset)
MaxAbs_scaler.fit(X_train_GloVe)
X_train_GloVe_scaled = MaxAbs_scaler.transform(X_train_GloVe)
X_testN0_GloVe_scaled = MaxAbs_scaler.transform(X_testN0_GloVe)
X_testN1_GloVe_scaled = MaxAbs_scaler.transform(X_testN1_GloVe)
X_testN2_GloVe_scaled = MaxAbs_scaler.transform(X_testN2_GloVe)
X_testN3_GloVe_scaled = MaxAbs_scaler.transform(X_testN3_GloVe)

#prior to this block: 8.0 gb
#after: 10.6 (temp 14.2)

In [ ]:
#shift +1 for all non-zero values to be usable for Naive Bayes classifier
#from scipy.sparse.csc import csc_matrix
X_train_GloVe_scaled += 1
X_testN0_GloVe_scaled += 1
X_testN1_GloVe_scaled += 1
X_testN2_GloVe_scaled += 1
X_testN3_GloVe_scaled += 1

# Embedding 5: word2vec

In [ ]:
#EMBEDDING_INDEX
#Creating the.... Embedding Index (aka embedding dictionary)

embedding_dim = 50 #based on ...file..  ('...50d.txt')
embeddings_index = {} # == 'embeddings_dictionary' in https://www.kaggle.com/code/andreshg/nlp-glove-bert-tf-idf-lstm-explained#5.-Vectorization

with gzip.open('/content/drive/MyDrive/MYDATA/Embeddings_PreTrained/word2vec/w2v_PCA_reduced-vectors.txt.gz', 'r') as f:
    for line in f:
        values = line.split()
        word = values[0].decode('utf-8')
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs


print(f'Found {len(embeddings_index)} word vectors.')

In [ ]:
#EMBEDDING_MATRIX
#this is where we need to grab each word from the padded sequences and match them with the embeddings index to create the embedding_matrix
embedding_matrix = np.zeros((vocab_length, embedding_dim))

for word, embedding in word_tokenizer.word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[embedding] = embedding_vector
        
#embedding_matrix #numpy array

In [ ]:
# replacing the above 5 blocks:
# For GloVe this block took about 10 GB
"""
def embedding_generator(X, embedding_matrix):
    for x in X:
        yield embedding_matrix[x]
"""
# training set
nparraylist_train = np.array(list(embedding_generator(X_train_SeqPad, embedding_matrix)))

# test sets
test_sets = [X_testN0_SeqPad, X_testN1_SeqPad, X_testN2_SeqPad, X_testN3_SeqPad]
nparraylist_tests = []
for X_test in test_sets:
    nparraylist_tests.append(np.array(list(embedding_generator(X_test, embedding_matrix))))

# reshape arrays
dims = (len(nparraylist_train), len(nparraylist_train[0]) * len(nparraylist_train[0][0]))
X_train_w2v = np.reshape(nparraylist_train, dims)

X_tests_w2v = []
for nparraylist_test in nparraylist_tests:
    dims = (len(nparraylist_test), len(nparraylist_test[0]) * len(nparraylist_test[0][0]))
    X_tests_w2v.append(np.reshape(nparraylist_test, dims))

X_testN0_w2v, X_testN1_w2v, X_testN2_w2v, X_testN3_w2v = X_tests_w2v

In [ ]:
#np.array_equal(X_testN0_w2v, X_testN1_w2v, equal_nan=False) #False, so that seems correct
#np.array_equal(X_testN0_GloVe, X_testN0_w2v, equal_nan=False) #False, so that seems correct

In [ ]:
#THIS BLOCK IS ONLY FOR THE NAIVE BAYES CLASSIFIER

#scaling
#minmax would be the go to for sparse data, however our sparse data contains negative values -> MaxAbsScaler
#from sklearn.preprocessing import MaxAbsScaler
#THis takes another 10GB...

MaxAbs_scaler = MaxAbsScaler()

# transform data
# (check whether indeed best to fit scaler only on trainset)
MaxAbs_scaler.fit(X_train_w2v)
X_train_w2v_scaled = MaxAbs_scaler.transform(X_train_w2v)
X_testN0_w2v_scaled = MaxAbs_scaler.transform(X_testN0_w2v)
X_testN1_w2v_scaled = MaxAbs_scaler.transform(X_testN1_w2v)
X_testN2_w2v_scaled = MaxAbs_scaler.transform(X_testN2_w2v)
X_testN3_w2v_scaled = MaxAbs_scaler.transform(X_testN3_w2v)

In [ ]:
#shift +1 for all non-zero values to be usable for Naive Bayes classifier
#from scipy.sparse.csc import csc_matrix
X_train_w2v_scaled += 1
X_testN0_w2v_scaled += 1
X_testN1_w2v_scaled += 1
X_testN2_w2v_scaled += 1
X_testN3_w2v_scaled += 1

# Embedding 6: FastText

In [ ]:
#EMBEDDING_INDEX
#Creating the.... Embedding Index (aka embedding dictionary)

embedding_dim = 50 #based on ...file..  ('...50d.txt')
embeddings_index = {} # == 'embeddings_dictionary'

with gzip.open('/content/drive/MyDrive/MYDATA/Embeddings_PreTrained/FastText/FastText_PCA_reduced-vectors.txt.gz', 'r') as f:
    for line in f:
        values = line.split()
        word = values[0].decode('utf-8')
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs


print(f'Found {len(embeddings_index)} word vectors.')

In [ ]:
#EMBEDDING_MATRIX
#this is where we need to grab each word from the padded sequences and match them with the embeddings index to create the embedding_matrix
embedding_matrix = np.zeros((vocab_length, embedding_dim))

for word, embedding in word_tokenizer.word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[embedding] = embedding_vector
        
#embedding_matrix #numpy array

In [ ]:
# For GloVe this block took about 10 GB
"""
def embedding_generator(X, embedding_matrix):
    for x in X:
        yield embedding_matrix[x]
"""
# training set
nparraylist_train = np.array(list(embedding_generator(X_train_SeqPad, embedding_matrix)))

# test sets
test_sets = [X_testN0_SeqPad, X_testN1_SeqPad, X_testN2_SeqPad, X_testN3_SeqPad]
nparraylist_tests = []
for X_test in test_sets:
    nparraylist_tests.append(np.array(list(embedding_generator(X_test, embedding_matrix))))

# reshape arrays
dims = (len(nparraylist_train), len(nparraylist_train[0]) * len(nparraylist_train[0][0]))
X_train_ft = np.reshape(nparraylist_train, dims)

X_tests_ft = []
for nparraylist_test in nparraylist_tests:
    dims = (len(nparraylist_test), len(nparraylist_test[0]) * len(nparraylist_test[0][0]))
    X_tests_ft.append(np.reshape(nparraylist_test, dims))

X_testN0_ft, X_testN1_ft, X_testN2_ft, X_testN3_ft = X_tests_ft

In [ ]:
#THIS BLOCK IS ONLY FOR THE NAIVE BAYES CLASSIFIER

#scaling
#minmax would be the go to for sparse data, however our sparse data contains negative values -> MaxAbsScaler
#from sklearn.preprocessing import MaxAbsScaler
#THis takes another 10GB...

MaxAbs_scaler = MaxAbsScaler()

# transform data
# (check whether indeed best to fit scaler only on trainset)
MaxAbs_scaler.fit(X_train_ft)
X_train_ft_scaled = MaxAbs_scaler.transform(X_train_ft)
X_testN0_ft_scaled = MaxAbs_scaler.transform(X_testN0_ft)
X_testN1_ft_scaled = MaxAbs_scaler.transform(X_testN1_ft)
X_testN2_ft_scaled = MaxAbs_scaler.transform(X_testN2_ft)
X_testN3_ft_scaled = MaxAbs_scaler.transform(X_testN3_ft)

In [ ]:
#shift +1 for all non-zero values to be usable for Naive Bayes classifier
#from scipy.sparse.csc import csc_matrix
X_train_ft_scaled += 1
X_testN0_ft_scaled += 1
X_testN1_ft_scaled += 1
X_testN2_ft_scaled += 1
X_testN3_ft_scaled += 1

In [ ]:
nparraylist_train, nparraylist_test, nparraylist_testN0, nparraylist_testN1, nparraylist_testN2, nparraylist_testN3 = [], [], [], [], [], []
del nparraylist_train, nparraylist_testN0, nparraylist_testN1, nparraylist_testN2, nparraylist_testN3

In [ ]:
gc.collect()

# Check RAM

In [ ]:
# also see del statement under glove embedding code
# and the gc collect

In [ ]:
#code snippet to list all self defined vars and their RAM uptake
#source: https://stackoverflow.com/questions/40993626/list-memory-usage-in-ipython-and-jupyter

#import sys

# These are the usual ipython objects, including this one you are creating
ipython_vars = ['In', 'Out', 'exit', 'quit', 'get_ipython', 'ipython_vars']

# Get a sorted list of the objects and their sizes
sorted([(x, sys.getsizeof(globals().get(x))) for x in dir() if not x.startswith('_') and x not in sys.modules and x not in ipython_vars], key=lambda x: x[1], reverse=True)

# Model 0: Creating lists for iterative model comparisons.

In [ ]:
#ListTrainsetEmbeds = [X_train_BoW, X_train_HV, X_train_TFIDF, X_train_GloVe]
#ListModelNames_LR = ['Model_LR_BoW', 'Model_LR_HV', 'Model_LR_TFIDF', 'Model_LR_GloVe']

ListTestsets_BoW = [X_testN0_BoW, X_testN1_BoW, X_testN2_BoW, X_testN3_BoW]
ListTestsets_HV = [X_testN0_HV, X_testN1_HV, X_testN2_HV, X_testN3_HV]
ListTestsets_TFIDF = [X_testN0_TFIDF, X_testN1_TFIDF, X_testN2_TFIDF, X_testN3_TFIDF]
ListTestsets_GloVe = [X_testN0_GloVe, X_testN1_GloVe, X_testN2_GloVe, X_testN3_GloVe]
ListTestsets_w2v = [X_testN0_w2v, X_testN1_w2v, X_testN2_w2v, X_testN3_w2v]
ListTestsets_ft = [X_testN0_ft, X_testN1_ft, X_testN2_ft, X_testN3_ft]

ListTestsets_HV_scaled = [X_testN0_HV_scaled, X_testN1_HV_scaled, X_testN2_HV_scaled, X_testN3_HV_scaled]
ListTestsets_GloVe_scaled = [X_testN0_GloVe_scaled, X_testN1_GloVe_scaled, X_testN2_GloVe_scaled, X_testN3_GloVe_scaled]
ListTestsets_w2v_scaled = [X_testN0_w2v_scaled, X_testN1_w2v_scaled, X_testN2_w2v_scaled, X_testN3_w2v_scaled]
ListTestsets_ft_scaled = [X_testN0_ft_scaled, X_testN1_ft_scaled, X_testN2_ft_scaled, X_testN3_ft_scaled]

names_list = ['X_testN0', 'X_testN1', 'X_testN2', 'X_testN3']

# Model 1: Logistics Regression

In [ ]:
#from sklearn.linear_model import LogisticRegression
classifierLR = LogisticRegression(solver='lbfgs', max_iter=2000)

In [ ]:
#Training the models and pickling each one separately

Model_LR_BoW = classifierLR.fit(X_train_BoW, y_train)
with open(picklepath+"Model_LR_BoW.pkl", "wb") as f:
    dump(Model_LR_BoW, f)

Model_LR_HV = classifierLR.fit(X_train_HV, y_train)
with open(picklepath+"Model_LR_HV.pkl", "wb") as f:
    dump(Model_LR_HV, f)

Model_LR_TFIDF = classifierLR.fit(X_train_TFIDF, y_train)
with open(picklepath+"Model_LR_TFIDF.pkl", "wb") as f:
    dump(Model_LR_TFIDF, f)

Model_LR_GloVe = classifierLR.fit(X_train_GloVe, y_train)
with open(picklepath+"Model_LR_GloVe.pkl", "wb") as f:
    dump(Model_LR_GloVe, f)

In [ ]:
Model_LR_w2v= classifierLR.fit(X_train_w2v, y_train)
with open(picklepath+"Model_LR_w2v.pkl", "wb") as f:
    dump(Model_LR_w2v, f)

In [ ]:
Model_LR_ft= classifierLR.fit(X_train_ft, y_train)
with open(picklepath+"Model_LR_ft.pkl", "wb") as f:
    dump(Model_LR_ft, f)

In [ ]:
#checking datatset shapes:
# --- 
#X_train.shape # (67625,)
#X_train_xxxx -> are all (67625, 60326) except X_train_GloVe with 67625 by 5000
#X_train_BoW.shape == X_train_TFIDF.shape == X_train_HV.shape
#X_train_GloVe.shape #(67625, 5000)

In [ ]:
#no longer all the same shape... :
#X_train_BoW.shape # (67625, 53921)
#X_train_TFIDF.shape # (67625, 53921)
#X_train_HV.shape # (67625, 60326)

Comparing results Logistic regression with BoW. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_LR_BoW.pkl', 'rb') as f:
  Model_LR_BoW = load(f)
  score_test = Model_LR_BoW.score(X_train_BoW, y_train)
  print(f'Model overfit check (BoW + LR): {score_test}') #.9966..
  for i, name in zip(ListTestsets_BoW, names_list):
    y_pred = Model_LR_BoW.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)    
    #score_test = Model_LR_BoW.score(i, y_testALL)
    print(f'AUC score for BoW + LogisticRegression on {name} (1%split): {score_test}')

Comparing results Logistic regression with HashingVectorizer.

(N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath+ 'Model_LR_HV.pkl', 'rb') as f:
  Model_LR_HV = load(f)
  score_test = Model_LR_HV.score(X_train_HV, y_train)
  print(f'Model overfit check (HV + LR): {score_test}') #.9216..
  for i, name in zip(ListTestsets_HV, names_list):
    y_pred = Model_LR_HV.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred) 
    #score_test = Model_LR_HV.score(i, y_testALL)
    print(f'AUC score for HV + LogisticRegression on {name} (1%split): {score_test}')

Comparing results Logistic regression with TFIDF. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_LR_TFIDF.pkl', 'rb') as f:
  Model_LR_TFIDF = load(f)
  score_test = Model_LR_TFIDF.score(X_train_TFIDF, y_train)
  print(f'Model overfit check (TFIDF + LR): {score_test}') #.9510..
  for i, name in zip(ListTestsets_TFIDF, names_list):
    y_pred = Model_LR_TFIDF.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_LR_TFIDF.score(i, y_testALL)
    print(f'AUC score for TFIDF + LogisticRegression on {name} (1%split): {score_test}')

Comparing results Logistic regression with GloVe.

(N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_LR_GloVe.pkl', 'rb') as f:
  Model_LR_GloVe = load(f)
  score_test = Model_LR_GloVe.score(X_train_GloVe, y_train)
  print(f'Model overfit check (GloVe + LR): {score_test}') #.8634..
  for i, name in zip(ListTestsets_GloVe, names_list):
    y_pred = Model_LR_GloVe.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_LR_GloVe.score(i, y_testALL)
    print(f'AUC score for GloVe+ LogisticRegression on {name} (1%split): {score_test}')

Comparing results Logistic regression with Word2Vec.

(N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_LR_w2v.pkl', 'rb') as f:
  Model_LR_w2v = load(f)
  score_test = Model_LR_w2v.score(X_train_w2v, y_train)
  print(f'Model overfit check (w2v + LR): {score_test}') #.9966..
  for i, name in zip(ListTestsets_w2v, names_list):
    y_pred = Model_LR_w2v.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_LR_w2v.score(i, y_testALL)
    print(f'AUC score for w2v + LogisticRegression on {name} (1%split): {score_test}')

Comparing results Logistic regression with FastText.

(N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_LR_ft.pkl', 'rb') as f:
  Model_LR_ft = load(f)
  score_test = Model_LR_ft.score(X_train_w2v, y_train)
  print(f'Model overfit check (fasttext + LR): {score_test}') #.9966..
  for i, name in zip(ListTestsets_ft, names_list):
    y_pred = Model_LR_ft.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_LR_ft.score(i, y_testALL)
    print(f'AUC score for fasttext + LogisticRegression on {name} (1%split): {score_test}')

# Model 2: Naive Bayes

In [ ]:
#why MultinomialNB instead of Gaussian? GaussianNN seemed to need dense matrix as input and that kept overloading RAM
#as train set is about 60k by 67k which is fine for sparse but giant in dense matrix (see code at the bottom with failed attempts to use .todense function).
#from sklearn.naive_bayes import MultinomialNB
classifierNB = MultinomialNB()

In [ ]:
#Training the models and pickling each one separately

Model_NB_BoW = classifierNB.fit(X_train_BoW, y_train, sample_weight=None)
with open(picklepath+"Model_NB_BoW.pkl", "wb") as f:
    dump(Model_NB_BoW, f)

Model_NB_HV = classifierNB.fit(X_train_HV_scaled, y_train, sample_weight=None)
with open(picklepath+"Model_NB_HV.pkl", "wb") as f:
    dump(Model_NB_HV, f)

Model_NB_TFIDF = classifierNB.fit(X_train_TFIDF, y_train, sample_weight=None)
with open(picklepath+"Model_NB_TFIDF.pkl", "wb") as f:
    dump(Model_NB_TFIDF, f)

Model_NB_GloVe = classifierNB.fit(X_train_GloVe_scaled, y_train, sample_weight=None)
with open(picklepath+"Model_NB_GloVe.pkl", "wb") as f:
    dump(Model_NB_GloVe, f)

In [ ]:
Model_NB_w2v = classifierNB.fit(X_train_w2v_scaled, y_train, sample_weight=None)
with open(picklepath+"Model_NB_w2v.pkl", "wb") as f:
    dump(Model_NB_w2v, f)

In [ ]:
Model_NB_ft = classifierNB.fit(X_train_ft_scaled, y_train, sample_weight=None)
with open(picklepath+"Model_NB_ft.pkl", "wb") as f:
    dump(Model_NB_ft, f)

Comparing results Naive Bayes with BoW. 

- (N0 (base), N1, N2 and N3)




In [ ]:
with open(picklepath + 'Model_NB_BoW.pkl', 'rb') as f:
  Model_NB_BoW = load(f)
  score_test = Model_NB_BoW.score(X_train_BoW, y_train)
  print(f'Model overfit check (BoW + NB): {score_test}') #.8730..
  for i, name in zip(ListTestsets_BoW, names_list):
    y_pred = Model_NB_BoW.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_NB_BoW.score(i, y_testALL)
    print(f'AUC score for BoW + Naive Bayes on {name} (1%split): {score_test}')

Comparing results Naive Bayes with HV. 

- (N0 (base), N1, N2 and N3)


In [ ]:
#this block uses the scaled input (MaxAbs scaler with a +1 shift for all nonzero values..)
with open(picklepath + 'Model_NB_HV.pkl', 'rb') as f:
  Model_NB_HV = load(f)
  score_test = Model_NB_HV.score(X_train_HV_scaled, y_train)
  print(f'Model overfit check (HV + NB): {score_test}') #.8682..
  for i, name in zip(ListTestsets_HV_scaled, names_list):
    y_pred = Model_NB_HV.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_NB_HV.score(i, y_testALL)
    print(f'AUC score for HashingVectors + Naive Bayes on {name} (1%split): {score_test}')

Comparing results Naive Bayes with TFIDF. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_NB_TFIDF.pkl', 'rb') as f:
  Model_NB_TFIDF = load(f)
  score_test = Model_NB_TFIDF.score(X_train_TFIDF, y_train)
  print(f'Model overfit check (TFIDF + NB): {score_test}') #.8753..
  for i, name in zip(ListTestsets_TFIDF, names_list):
    y_pred = Model_NB_TFIDF.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_NB_TFIDF.score(i, y_testALL)
    print(f'AUC score for TFIDF + Naive Bayes on {name} (1%split): {score_test}')

Comparing results Naive Bayes with GloVe.

(N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_NB_GloVe.pkl', 'rb') as f:
  Model_NB_GloVe = load(f)
  score_test = Model_NB_GloVe.score(X_train_GloVe_scaled, y_train)
  print(f'Model overfit check (GloVe + NB): {score_test}')
  for i, name in zip(ListTestsets_GloVe_scaled, names_list):
    y_pred = Model_NB_GloVe.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_NB_GloVe.score(i, y_testALL)
    print(f'AUC score for GloVe + Naive Bayes on {name} (1%split): {score_test}')
#Remember this uses the scaled and +1 shifted embeddings..

Comparing results Naive Bayes with Word2Vec.

(N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_NB_w2v.pkl', 'rb') as f:
  Model_NB_w2v = load(f)
  score_test = Model_NB_w2v.score(X_train_w2v_scaled, y_train)
  print(f'Model overfit check (w2v + NB): {score_test}')
  for i, name in zip(ListTestsets_w2v_scaled, names_list):
    y_pred = Model_NB_w2v.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_NB_w2v.score(i, y_testALL)
    print(f'AUC score for w2v + Naive Bayes on {name} (1%split): {score_test}')

Comparing results Naive Bayes with FastText.

(N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_NB_ft.pkl', 'rb') as f:
  Model_NB_ft = load(f)
  score_test = Model_NB_ft.score(X_train_ft_scaled, y_train)
  print(f'Model overfit check (FastText + NB): {score_test}')
  for i, name in zip(ListTestsets_ft_scaled, names_list):
    y_pred = Model_NB_ft.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_NB_ft.score(i, y_testALL)
    print(f'AUC score for FastText + Naive Bayes on {name} (1%split): {score_test}')

Check some more MN Naive Bayes evaluation code here:
https://www.kaggle.com/code/ibrahimaptlo10/eda-multi-class-multinomial-naive-bayes

# Model 3: Random Forest




In [ ]:
#from sklearn.ensemble import RandomForestClassifier
classifierRF = RandomForestClassifier(n_estimators= 18, max_depth = 8, min_samples_split=3)

In [ ]:
#still to do:
# add n_estimators? and ?max_features? etc to prevent overfitting.. 
# Prioritize speed of training over precision for now! (running this now takes about 15 minutes, with n_estimators at 20, this is about 2 minutes)

In [ ]:
#Training the models and pickling each one separately

Model_RF_BoW = classifierRF.fit(X_train_BoW, y_train)
with open(picklepath+"Model_RF_BoW.pkl", "wb") as f:
    dump(Model_RF_BoW, f)

Model_RF_HV = classifierRF.fit(X_train_HV_scaled, y_train)
with open(picklepath+"Model_RF_HV.pkl", "wb") as f:
    dump(Model_RF_HV, f)
# -> Negative values in data passed to MultinomialNB (input X)

Model_RF_TFIDF = classifierRF.fit(X_train_TFIDF, y_train)
with open(picklepath+"Model_RF_TFIDF.pkl", "wb") as f:
    dump(Model_RF_TFIDF, f)

Model_RF_GloVe = classifierRF.fit(X_train_GloVe_scaled, y_train)
with open(picklepath+"Model_RF_GloVe.pkl", "wb") as f:
    dump(Model_RF_GloVe, f)

In [ ]:
Model_RF_w2v = classifierRF.fit(X_train_w2v, y_train)
with open(picklepath+"Model_RF_w2v.pkl", "wb") as f:
    dump(Model_RF_w2v, f)

In [ ]:
Model_RF_ft = classifierRF.fit(X_train_ft, y_train)
with open(picklepath+"Model_RF_ft.pkl", "wb") as f:
    dump(Model_RF_ft, f)

Comparing results Random Forest with BoW. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_RF_BoW.pkl', 'rb') as f:
  Model_RF_BoW = load(f)
  score_test = Model_RF_BoW.score(X_train_BoW, y_train)
  print(f'Model overfit check (BoW + RF): {score_test}') #.9994..
  for i, name in zip(ListTestsets_BoW, names_list):
    y_pred = Model_RF_BoW.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_RF_BoW.score(i, y_testALL)
    print(f'AUC score for BoW + Random Forest on {name} (1%split): {score_test}')

Comparing results Random Forest with HV. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_RF_HV.pkl', 'rb') as f:
  Model_RF_HV = load(f)
  score_test = Model_RF_HV.score(X_train_HV_scaled, y_train)
  print(f'Model overfit check (HV + RF): {score_test}') #.9994..
  for i, name in zip(ListTestsets_HV_scaled, names_list):
    y_pred = Model_RF_HV.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_RF_HV.score(i, y_testALL)
    print(f'AUC score for HV + Random Forest on {name} (1%split): {score_test}')

Comparing results Random Forest with TFIDF. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_RF_TFIDF.pkl', 'rb') as f:
  Model_RF_TFIDF = load(f)
  score_test = Model_RF_TFIDF.score(X_train_TFIDF, y_train)
  print(f'Model overfit check (TFIDF + RF): {score_test}') #.9995..
  for i, name in zip(ListTestsets_TFIDF, names_list):
    y_pred = Model_RF_TFIDF.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_RF_TFIDF.score(i, y_testALL)
    print(f'AUC score for TFIDF + Random Forest on {name} (1%split): {score_test}')

Comparing results Random Forest with GloVe. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_RF_GloVe.pkl', 'rb') as f:
  Model_RF_GloVe = load(f)
  score_test = Model_RF_GloVe.score(X_train_GloVe_scaled, y_train)
  print(f'Model overfit check (GloVe + RF): {score_test}') #.9908..
  for i, name in zip(ListTestsets_GloVe_scaled, names_list):
    y_pred = Model_RF_GloVe.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_RF_GloVe.score(i, y_testALL)
    print(f'AUC score for GloVe + Random Forest on {name} (1%split): {score_test}')

Comparing results Random Forest with Word2Vec. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_RF_w2v.pkl', 'rb') as f:
  Model_RF_w2v = load(f)
  score_test = Model_RF_w2v.score(X_train_w2v, y_train)
  print(f'Model overfit check (w2v + RF): {score_test}') #.9994..
  for i, name in zip(ListTestsets_w2v, names_list):
    y_pred = Model_RF_w2v.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_RF_w2v.score(i, y_testALL)
    print(f'AUC score for w2v + Random Forest on {name} (1%split): {score_test}')

Comparing results Random Forest with FastText. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_RF_ft.pkl', 'rb') as f:
  Model_RF_ft = load(f)
  score_test = Model_RF_ft.score(X_train_ft, y_train)
  print(f'Model overfit check (ft + RF): {score_test}') #.9994..
  for i, name in zip(ListTestsets_ft, names_list):
    y_pred = Model_RF_ft.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_RF_ft.score(i, y_testALL)
    print(f'AUC score for ft + Random Forest on {name} (1%split): {score_test}')

# Model 4: SupportVectorMachine

In [ ]:
#from sklearn.linear_model import SGDClassifier
classifierSVM = SGDClassifier(loss='hinge', penalty='l2', random_state=42, alpha=0.0015)

In [ ]:
#Training the models and pickling each one separately

Model_SVM_BoW = classifierSVM.fit(X_train_BoW, y_train)
with open(picklepath+"Model_SVM_BoW.pkl", "wb") as f:
    dump(Model_SVM_BoW, f)

Model_SVM_HV = classifierSVM.fit(X_train_HV_scaled, y_train)
with open(picklepath+"Model_SVM_HV.pkl", "wb") as f:
    dump(Model_SVM_HV, f)

Model_SVM_TFIDF = classifierSVM.fit(X_train_TFIDF, y_train)
with open(picklepath+"Model_SVM_TFIDF.pkl", "wb") as f:
    dump(Model_SVM_TFIDF, f)

Model_SVM_GloVe = classifierSVM.fit(X_train_GloVe_scaled, y_train)
with open(picklepath+"Model_SVM_GloVe.pkl", "wb") as f:
    dump(Model_SVM_GloVe, f)

In [ ]:
Model_SVM_w2v = classifierSVM.fit(X_train_w2v, y_train)
with open(picklepath+"Model_SVM_w2v.pkl", "wb") as f:
    dump(Model_SVM_w2v, f)

In [ ]:
Model_SVM_ft = classifierSVM.fit(X_train_ft, y_train)
with open(picklepath+"Model_SVM_ft.pkl", "wb") as f:
    dump(Model_SVM_ft, f)

Comparing results SVM with BoW. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_SVM_BoW.pkl', 'rb') as f:
  Model_SVM_BoW = load(f)
  score_test = Model_SVM_BoW.score(X_train_BoW, y_train)
  print(f'Model overfit check (BoW + SVM): {score_test}') #.9876..
  for i, name in zip(ListTestsets_BoW, names_list):
    y_pred = Model_SVM_BoW.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_SVM_BoW.score(i, y_testALL)
    print(f'AUC score for BoW + SVM on {name} (1%split): {score_test}')

Comparing results SVM with HV. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_SVM_HV.pkl', 'rb') as f:
  Model_SVM_HV = load(f)
  score_test = Model_SVM_HV.score(X_train_HV_scaled, y_train)
  print(f'Model overfit check (HV + SVM): {score_test}') #.9110..
  for i, name in zip(ListTestsets_HV_scaled, names_list):
    y_pred = Model_SVM_HV.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_SVM_HV.score(i, y_testALL)
    print(f'AUC score for HashingVectors + SVM on {name} (1%split): {score_test}')

Comparing results SVM with TFIDF. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_SVM_TFIDF.pkl', 'rb') as f:
  Model_SVM_TFIDF = load(f)
  score_test = Model_SVM_TFIDF.score(X_train_TFIDF, y_train)
  print(f'Model overfit check (TFIDF + SVM): {score_test}') #.9408..
  for i, name in zip(ListTestsets_TFIDF, names_list):
    y_pred = Model_SVM_TFIDF.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_SVM_TFIDF.score(i, y_testALL)
    print(f'AUC score for TFIDF + SVM on {name} (1%split): {score_test}')

Comparing results SVM with GloVe. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_SVM_GloVe.pkl', 'rb') as f:
  Model_SVM_GloVe = load(f)
  score_test = Model_SVM_GloVe.score(X_train_GloVe_scaled, y_train)
  print(f'Model overfit check (TFIDF + SVM): {score_test}') #.7574..
  for i, name in zip(ListTestsets_GloVe_scaled, names_list):
    y_pred = Model_SVM_GloVe.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_SVM_GloVe.score(i, y_testALL)
    print(f'AUC score for GloVe + SVM on {name} (1%split): {score_test}')

Comparing results SVM with Word2Vec. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_SVM_w2v.pkl', 'rb') as f:
  Model_SVM_w2v = load(f)
  score_test = Model_SVM_w2v.score(X_train_w2v, y_train)
  print(f'Model overfit check (w2v + SVM): {score_test}') #.7574..
  for i, name in zip(ListTestsets_w2v, names_list):
    y_pred = Model_SVM_w2v.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_SVM_w2v.score(i, y_testALL)
    print(f'AUC score for w2v + SVM on {name} (1%split): {score_test}')

Comparing results SVM with FastText. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_SVM_ft.pkl', 'rb') as f:
  Model_SVM_ft = load(f)
  score_test = Model_SVM_ft.score(X_train_ft, y_train)
  print(f'Model overfit check (ft + SVM): {score_test}') #.7574..
  for i, name in zip(ListTestsets_ft, names_list):
    y_pred = Model_SVM_ft.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_SVM_ft.score(i, y_testALL)
    print(f'AUC score for FastText + SVM on {name} (1%split): {score_test}')

# Model 5: K NearestNeighbour (KNN)

In [ ]:
classifierKNN = KNeighborsClassifier(n_neighbors=6, leaf_size=8, weights='uniform')
#remember: training KNN is fast, scoring is slow and RAM intensive.
#also.. these have giant pickle models (.pkl's) so change parameters a bit here..

In [ ]:
#Training the models and pickling each one separately

Model_KNN_BoW = classifierKNN.fit(X_train_BoW, y_train)
with open(picklepath+"Model_KNN_BoW.pkl", "wb") as f:
    dump(Model_KNN_BoW, f)

Model_KNN_HV = classifierKNN.fit(X_train_HV, y_train)
with open(picklepath+"Model_KNN_HV.pkl", "wb") as f:
    dump(Model_KNN_HV, f)

Model_KNN_TFIDF = classifierKNN.fit(X_train_TFIDF, y_train)
with open(picklepath+"Model_KNN_TFIDF.pkl", "wb") as f:
    dump(Model_KNN_TFIDF, f)

Model_KNN_GloVe = classifierKNN.fit(X_train_GloVe, y_train)
with open(picklepath+"Model_KNN_GloVe.pkl", "wb") as f:
    dump(Model_KNN_GloVe, f)

In [ ]:
Model_KNN_w2v = classifierKNN.fit(X_train_w2v, y_train)
with open(picklepath+"Model_KNN_w2v.pkl", "wb") as f:
    dump(Model_KNN_w2v, f)

In [ ]:
Model_KNN_ft = classifierKNN.fit(X_train_ft, y_train)
with open(picklepath+"Model_KNN_ft.pkl", "wb") as f:
    dump(Model_KNN_ft, f)

Comparing results KNN with BoW. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_KNN_BoW.pkl', 'rb') as f:
  Model_KNN_BoW = load(f)
  score_test = Model_KNN_BoW.score(X_train_BoW, y_train)
  print(f'Model overfit check (BoW + KNN): {score_test}') #.9096..
  for i, name in zip(ListTestsets_BoW, names_list):
    y_pred = Model_KNN_BoW.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_KNN_BoW.score(i, y_testALL)
    print(f'AUC score for BoW + KNN on {name} (1%split): {score_test}')

Comparing results KNN with HV. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_KNN_HV.pkl', 'rb') as f:
  Model_KNN_HV = load(f)
  score_test = Model_KNN_HV.score(X_train_HV, y_train)
  print(f'Model overfit check (HV + KNN): {score_test}') #.8668..
  for i, name in zip(ListTestsets_HV, names_list):
    y_pred = Model_KNN_HV.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_KNN_HV.score(i, y_testALL)
    print(f'AUC score for HashingVectors + KNN on {name} (1%split): {score_test}')

Comparing results KNN with TFIDF. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_KNN_TFIDF.pkl', 'rb') as f:
  Model_KNN_TFIDF = load(f)
  score_test = Model_KNN_TFIDF.score(X_train_TFIDF, y_train)
  print(f'Model overfit check (TFIDF + KNN): {score_test}') #.8907..
  for i, name in zip(ListTestsets_TFIDF, names_list):
    y_pred = Model_KNN_TFIDF.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_KNN_TFIDF.score(i, y_testALL)
    print(f'AUC score for TFIDF + KNN on {name} (1%split): {score_test}')

Comparing results KNN with GloVe. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_KNN_GloVe.pkl', 'rb') as f:
  Model_KNN_GloVe = load(f)
  score_test = Model_KNN_GloVe.score(X_train_GloVe, y_train)
  print(f'Model overfit check (GloVe + KNN): {score_test}') #.8720..
  for i, name in zip(ListTestsets_GloVe, names_list):
    y_pred = Model_KNN_GloVe.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_KNN_GloVe.score(i, y_testALL)
    print(f'AUC score for GloVe + KNN on {name} (1%split): {score_test}')

Comparing results KNN with Word2Vec. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_KNN_w2v.pkl', 'rb') as f:
  Model_KNN_w2v = load(f)
  score_test = Model_KNN_w2v.score(X_train_w2v, y_train)
  print(f'Model overfit check (w2v + KNN): {score_test}') #.7574..
  for i, name in zip(ListTestsets_w2v, names_list):
    y_pred = Model_KNN_w2v.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_KNN_w2v.score(i, y_testALL)
    print(f'AUC score for w2v + KNN on {name} (1%split): {score_test}')

Comparing results KNN with FastText. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_KNN_ft.pkl', 'rb') as f:
  Model_KNN_ft = load(f)
  score_test = Model_KNN_ft.score(X_train_ft, y_train)
  print(f'Model overfit check (ft + KNN): {score_test}') #.7574..
  for i, name in zip(ListTestsets_ft, names_list):
    y_pred = Model_KNN_ft.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_KNN_ft.score(i, y_testALL)
    print(f'AUC score for FastText + KNN on {name} (1%split): {score_test}')

# Model 6. GradientBoosting

In [ ]:
#from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
classifierGB = GradientBoostingClassifier(learning_rate=0.3, max_depth=2)
#change from 0.1 to 0.2 to save time when testing.. (NOTE: takes long time still at .2..)

In [ ]:
#Training the models and pickling each one separately

Model_GB_BoW = classifierGB.fit(X_train_BoW, y_train)
with open(picklepath+"Model_GB_BoW.pkl", "wb") as f:
    dump(Model_GB_BoW, f)

Model_GB_HV = classifierGB.fit(X_train_HV, y_train)
with open(picklepath+"Model_GB_HV.pkl", "wb") as f:
    dump(Model_GB_HV, f)

Model_GB_TFIDF = classifierGB.fit(X_train_TFIDF, y_train)
with open(picklepath+"Model_GB_TFIDF.pkl", "wb") as f:
    dump(Model_GB_TFIDF, f)

Model_GB_GloVe = classifierGB.fit(X_train_GloVe, y_train)
with open(picklepath+"Model_GB_GloVe.pkl", "wb") as f:
    dump(Model_GB_GloVe, f)

In [ ]:
Model_GB_w2v = classifierGB.fit(X_train_w2v, y_train)
with open(picklepath+"Model_GB_w2v.pkl", "wb") as f:
    dump(Model_GB_w2v, f)

In [ ]:
Model_GB_ft = classifierGB.fit(X_train_ft, y_train)
with open(picklepath+"Model_GB_ft.pkl", "wb") as f:
    dump(Model_GB_ft, f)

Comparing results GradientBoosting with BoW. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_GB_BoW.pkl', 'rb') as f:
  Model_GB_BoW = load(f)
  score_test = Model_GB_BoW.score(X_train_BoW, y_train)
  print(f'Model overfit check (BoW + GB): {score_test}') #.9096..
  for i, name in zip(ListTestsets_BoW, names_list):
    y_pred = Model_GB_BoW.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_GB_BoW.score(i, y_testALL)
    print(f'AUC score for BoW + GB on {name} (1%split): {score_test}')

Comparing results GradientBoosting with HV. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_GB_HV.pkl', 'rb') as f:
  Model_GB_HV = load(f)
  score_test = Model_GB_HV.score(X_train_HV, y_train)
  print(f'Model overfit check (HV + GB): {score_test}') #.8668..
  for i, name in zip(ListTestsets_HV, names_list):
    y_pred = Model_GB_HV.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_GB_HV.score(i, y_testALL)
    print(f'AUC score for HashingVectors + GB on {name} (1%split): {score_test}')

Comparing results GradientBoosting with TFIDF. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_GB_TFIDF.pkl', 'rb') as f:
  Model_GB_TFIDF = load(f)
  score_test = Model_GB_TFIDF.score(X_train_TFIDF, y_train)
  print(f'Model overfit check (TFIDF + GB): {score_test}') #.8907..
  for i, name in zip(ListTestsets_TFIDF, names_list):
    y_pred = Model_GB_TFIDF.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_GB_TFIDF.score(i, y_testALL)
    print(f'AUC score for TFIDF + GB on {name} (1%split): {score_test}')

Comparing results GradientBoosting with GloVe. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_GB_GloVe.pkl', 'rb') as f:
  Model_GB_GloVe = load(f)
  score_test = Model_GB_GloVe.score(X_train_GloVe, y_train)
  print(f'Model overfit check (GloVe + GB): {score_test}') #.8720..
  for i, name in zip(ListTestsets_GloVe, names_list):
    y_pred = Model_GB_GloVe.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_GB_GloVe.score(i, y_testALL)
    print(f'AUC score for GloVe + GB on {name} (1%split): {score_test}')

Comparing results GradientBoosting with Word2Vec. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_GB_w2v.pkl', 'rb') as f:
  Model_GB_w2v = load(f)
  score_test = Model_GB_w2v.score(X_train_w2v, y_train)
  print(f'Model overfit check (w2v + GB): {score_test}') #.7574..
  for i, name in zip(ListTestsets_w2v, names_list):
    y_pred = Model_GB_w2v.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_GB_w2v.score(i, y_testALL)
    print(f'AUC score for word2vec + GB on {name} (1%split): {score_test}')

Comparing results GradientBoosting with FastText. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_GB_ft.pkl', 'rb') as f:
  Model_GB_ft = load(f)
  score_test = Model_GB_ft.score(X_train_ft, y_train)
  print(f'Model overfit check (ft + GB): {score_test}') #.7574..
  for i, name in zip(ListTestsets_ft, names_list):
    y_pred = Model_GB_ft.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_GB_ft.score(i, y_testALL)
    print(f'AUC score for FastText + GB on {name} (1%split): {score_test}')

# Model 7. XGBoost (extreme gradient boosting)

In [ ]:
#from xgboost import XGBClassifier

In [ ]:
classifierXGB = XGBClassifier(learning_rate=0.2, max_depth=3)

In [ ]:
#Training the models and pickling each one separately
Model_XGB_BoW = classifierXGB.fit(X_train_BoW, y_train)
with open(picklepath+"Model_XGB_BoW.pkl", "wb") as f:
    dump(Model_XGB_BoW, f)

Model_XGB_HV = classifierXGB.fit(X_train_HV, y_train)
with open(picklepath+"Model_XGB_HV.pkl", "wb") as f:
    dump(Model_XGB_HV, f)

Model_XGB_TFIDF = classifierXGB.fit(X_train_TFIDF, y_train)
with open(picklepath+"Model_XGB_TFIDF.pkl", "wb") as f:
    dump(Model_XGB_TFIDF, f)

Model_XGB_GloVe = classifierXGB.fit(X_train_GloVe, y_train)
with open(picklepath+"Model_XGB_GloVe.pkl", "wb") as f:
    dump(Model_XGB_GloVe, f)

In [ ]:
Model_XGB_w2v = classifierXGB.fit(X_train_w2v, y_train)
with open(picklepath+"Model_XGB_w2v.pkl", "wb") as f:
    dump(Model_XGB_w2v, f)

In [ ]:
Model_XGB_ft = classifierXGB.fit(X_train_ft, y_train)
with open(picklepath+"Model_XGB_ft.pkl", "wb") as f:
    dump(Model_XGB_ft, f)

Comparing results XGB with BoW. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_XGB_BoW.pkl', 'rb') as f:
  Model_XGB_BoW = load(f)
  score_test = Model_XGB_BoW.score(X_train_BoW, y_train)
  print(f'Model overfit check (BoW + XGB): {score_test}') #.9096..
  for i, name in zip(ListTestsets_BoW, names_list):
    y_pred = Model_XGB_BoW.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_XGB_BoW.score(i, y_testALL)
    print(f'AUC score for BoW + XGB on {name} (1%split): {score_test}')

Comparing results XGB with HV. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_XGB_HV.pkl', 'rb') as f:
  Model_XGB_HV = load(f)
  score_test = Model_XGB_HV.score(X_train_HV, y_train)
  print(f'Model overfit check (HV + XGB): {score_test}') #.8668..
  for i, name in zip(ListTestsets_HV, names_list):
    y_pred = Model_XGB_HV.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_XGB_HV.score(i, y_testALL)
    print(f'AUC score for HashingVectors + XGB on {name} (1%split): {score_test}')

Comparing results XGB with TFIDF. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_XGB_TFIDF.pkl', 'rb') as f:
  Model_XGB_TFIDF = load(f)
  score_test = Model_XGB_TFIDF.score(X_train_TFIDF, y_train)
  print(f'Model overfit check (TFIDF + XGB): {score_test}') #.8907..
  for i, name in zip(ListTestsets_TFIDF, names_list):
    y_pred = Model_XGB_TFIDF.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_XGB_TFIDF.score(i, y_testALL)
    print(f'AUC score for TFIDF + XGB on {name} (1%split): {score_test}')

Comparing results XGB with GloVe. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_XGB_GloVe.pkl', 'rb') as f:
  Model_XGB_GloVe = load(f)
  score_test = Model_XGB_GloVe.score(X_train_GloVe, y_train)
  print(f'Model overfit check (GloVe + XGB): {score_test}') #.8720..
  for i, name in zip(ListTestsets_GloVe, names_list):
    y_pred = Model_XGB_GloVe.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_XGB_GloVe.score(i, y_testALL)
    print(f'AUC score for GloVe + XGB on {name} (1%split): {score_test}')

Comparing results XGB with Word2Vec. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_XGB_w2v.pkl', 'rb') as f:
  Model_XGB_w2v = load(f)
  score_test = Model_XGB_w2v.score(X_train_w2v, y_train)
  print(f'Model overfit check (w2v + XGB): {score_test}') #.7574..
  for i, name in zip(ListTestsets_w2v, names_list):
    y_pred = Model_XGB_w2v.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_XGB_w2v.score(i, y_testALL)
    print(f'AUC score for word2vec + XGB on {name} (1%split): {score_test}')

Comparing results XGB with FastText. 

- (N0 (base), N1, N2 and N3)

In [ ]:
with open(picklepath + 'Model_XGB_ft.pkl', 'rb') as f:
  Model_XGB_ft = load(f)
  score_test = Model_XGB_ft.score(X_train_ft, y_train)
  print(f'Model overfit check (ft + XGB): {score_test}') #.7574..
  for i, name in zip(ListTestsets_ft, names_list):
    y_pred = Model_XGB_ft.predict(i)
    score_test = roc_auc_score(y_testALL, y_pred)
    #score_test = Model_XGB_ft.score(i, y_testALL)
    print(f'AUC score for FastText + XGB on {name} (1%split): {score_test}')

# Noise Proxy
- JaroWinkler (and Hamming) Distance as proxy for 'Noise'. 
- Calculated in separate notebook, imported from csv.

In [ ]:
pathJWdist = '/content/drive/MyDrive/MYDATA/LexicalDistance/LexD_JaroWinkler.csv'
pathHamdist = '/content/drive/MyDrive/MYDATA/LexicalDistance/LexD_Hamming.csv'

In [ ]:
df_JaroWinkler = pd.read_csv(pathJWdist)
df_Hamming = pd.read_csv(pathHamdist)

In [ ]:
#display(df_JaroWinkler.describe())

In [ ]:
#display(df_Hamming.describe())

In [ ]:
#values taken from the describe method commented out above

# line 1 points
x1 = [1,2,3]
y1 = [0.136735,0.101772,0.069857]
# plotting the line 1 points 
plt.plot(x1, y1, label = "Hamming Distance")
# line 2 points
x2 = [1,2,3]
y2 = [0.136735,0.077469,0.054566]
# plotting the line 2 points 
plt.plot(x2, y2, label = "Jaro-Winkler Distance")
plt.xlabel('back-translation')
# Set the y axis label of the current axis.
plt.ylabel('Mean of Doc Distance (normalized)')
# Set a title of the current axes.
plt.title('Hamming and JaroWinkler for lexixal distance between backtranslation levels')
# show a legend on the plot
plt.legend()
# Display a figure.
plt.show()

# AUC ROC's across noise levels

| Model/Alg              | Embedding/vectorizer  | N0 (JWD=0.000)      | N1 (JWD=0.137)      | N2 (JWD=0.141)     | N3 (JWD=0.142)      |
|------------------------|-----------------------|---------|---------|---------|---------|
| Logistic Regression    | BoW                   | .9077   | .8506   | .8487   | .8479   |
|                        | HV                    | .8891   | .8505   | .8481   | .8482   |
|                        | TFIDF                 | .9131   | .8705   | .8690   | .8679   |
|                        | GloVe                 | .7374   | .6985   | .7006   | .7005   |
|                        | Word2Vec (CPA_red)    | .7475   | .6997   | .6977   | .6962   |
|                        | FastText (CPA_red)    | .7649   | .7260   | .7269   | .7288   |
| Naive Bayes            | BoW                   | .8313   | .7936   | .7912   | .7905   |
|                        | HV                    | .8249   | .7800   | .7784   | .7787   |
|                        | TFIDF                 | .8484   | .8052   | .8041   | .8033   |
|                        | GloVe*                | .6221   | .6043   | .6026   | .6022   |
|                        | Word2Vec (CPA_red)*   | .6107   | .5827   | .5817   | .5813   |
|                        | FastText (CPA_red)*   | .5629   | .5434   | .5423   | .5429   |
| Random Forest          | BoW                   | .8102   | .7570   | .7572   | .7563   |
|                        | HV*                   | .7892   | .7492   | .7488   | .7457   |
|                        | TFIDF                 | .8086   | .7762   | .7746   | .7747   |
|                        | GloVe*                | .7226   | .6762   | .6696   | .6696   |
|                        | Word2Vec (CPA_red)    | .7247   | .6710   | .6659   | .6631   |
|                        | FastText (CPA_red)    | .7262   | .6659   | .6626   | .6582   |
| SVM                    | BoW                   | .9180   | .8604   | .8576   | .8574   |
|                        | HV                    | .9202   | .8610   | .8591   | .8571   |
|                        | TFIDF                 | .8767   | .8447   | .8437   | .8437   |
|                        | GloVe                 | .6839   | .6639   | .6627   | .6648   |
|                        | Word2Vec (CPA_red)    | .7791   | .7164   | .7163   | .7191   |
|                        | FastText (CPA_red)    | .7907   | .7429   | .7430   | .7452   |
| KNN                    | BoW                   | .7226   | .6921   | .6866   | .6869   |
|                        | HV                    | .7612   | .7357   | .7369   | .7374   |
|                        | TFIDF                 | .7962   | .7693   | .7683   | .7684   |
|                        | GloVe                 | .5971   | .5725   | .5723   | .5723   |
|                        | Word2Vec (CPA_red)    | .4992   | .4939   | .4931   | .4920   |
|                        | FastText (CPA_red)    | .5922   | .5689   | .5697   | .5681   |
| GradientBoosting       | BoW                   | .8947   | .8287   | .8258   | .8247   |
|                        | HV                    | .8956   | .8349   | .8340   | .8329   |       
|                        | TFIDF                 | .8948   | .8349   | .8321   | .8321   |       
|                        | GloVe                 | .7865   | .7313   | .7315   | .7300   |       
|                        | Word2Vec (CPA_red)    | .7862   | .7261   | .7238   | .7250   |
|                        | FastText (CPA_red)    | .7878   | .7338   | .7321   | .7307   |       |
| XGB                    | BoW                   | .8973   | .8357   | .8340   | .8337   |
|                        | HV                    | .8982   | .8424   | .8405   | .8386   |       
|                        | TFIDF                 | .8980   | .8383   | .8361  | .8350    |       
|                        | GloVe                 | .8010   | .7540   | .7490  | .7481    |       
|                        | Word2Vec (CPA_red)    | .8072   | .7475   | .7418   | .7413   |
|                        | FastText (CPA_red)    | .8057   | .7526   | .7476   | .7459   |       |
